# 07 - Basic RAG


## What this notebook is for

This notebook is written as **notes + code**. Read the Markdown cells first, then run the code cells.

The goal is to understand the concept, see the architecture, implement a small example, and then evaluate/improve it.

## Learning objectives

- Build the complete RAG pipeline.
- Load → split → embed → store → retrieve → generate.
- Preserve source metadata.
- Understand where hallucination can still happen.

## Core AI concept

Basic RAG is a two-part system: retrieval finds evidence; generation writes the answer. Retrieval does not automatically make an answer correct. The generator must also be instructed to stay grounded in the retrieved context.

## Architecture / mental model

```text
Documents → chunks → embeddings → vector DB
                                      ↓
Question → embedding → top-k retrieval → context → LLM → answer
```

## Connection to the supplied learning material

Your supplied `rag.ipynb` explicitly implements document loading, fixed-size chunking, embeddings, Chroma storage, retrieval, citation-ready context, generation, reranking, and a small RAG evaluation section.

### Source foundation
This notebook is an expanded learning version of the uploaded notebooks. The source material for this track includes:
- `chunking.ipynb`
- `embeddings.ipynb`
- `faiss.ipynb`
- `chroma_db.ipynb`
- `searching.ipynb`
- `rag.ipynb`

The original examples and terminology are retained where useful, then organized into a cleaner learning sequence.

## Setup

The code below uses environment variables for secrets and model names so the notebook is easier to maintain when providers or model versions change.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")
OPENAI_EMBED_MODEL = os.getenv("OPENAI_EMBED_MODEL", "text-embedding-3-small")

print("Model:", OPENAI_MODEL)
print("Embedding model:", OPENAI_EMBED_MODEL)

In [ ]:
%pip install -qU langchain-openai langchain-chroma langchain-text-splitters

### Build the corpus

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

raw_docs = [
    Document(page_content="RAG stands for Retrieval-Augmented Generation. It retrieves external information before generation.",
             metadata={"source": "rag.txt"}),
    Document(page_content="Python is widely used for data science, machine learning, automation, and AI.",
             metadata={"source": "python.txt"}),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)

for d in chunks:
    print(d.metadata, d.page_content)

### Embeddings + Chroma

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model=OPENAI_EMBED_MODEL)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="basic_rag_demo",
    persist_directory="./basic_rag_chroma",
)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

### Prompt + model

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    """You are a grounded assistant.
Use only the context below.

Context:
{context}

Question:
{question}

Answer with source names when possible. If the context is insufficient, say so.
"""
)

llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0)
parser = StrOutputParser()

def format_docs(documents):
    return "\n\n".join(
        f"[Source: {d.metadata.get('source', 'unknown')}]\n{d.page_content}"
        for d in documents
    )

def basic_rag(question):
    retrieved = retriever.invoke(question)
    context = format_docs(retrieved)
    answer = (prompt | llm | parser).invoke({
        "context": context,
        "question": question,
    })
    return answer, retrieved

# answer, docs = basic_rag("What is RAG?")
# print(answer)

## Evaluation

Create questions with expected sources. Measure whether the correct source appears in top-k and whether the answer is supported by those sources.

## Improvement ideas

Tune chunks and overlap first, then retrieval settings, then reranking, then the prompt/model. Diagnose the failed stage instead of changing everything at once.

## Practice exercises

1. Add source citations with chunk IDs.
2. Add a no-answer question.
3. Compare k=2, 4, and 8.

## Final checklist

- [ ] I can explain the concept in simple words.
- [ ] I can describe where it fits in a GenAI system.
- [ ] I can run and modify the code.
- [ ] I know at least one way to evaluate it.
- [ ] I know what to change when it fails.

## Research / reference notes


- LangChain Python reference: https://reference.langchain.com/python/langchain/overview
- LangGraph Python reference: https://reference.langchain.com/python/langgraph/overview
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- OpenAI function calling guide: https://developers.openai.com/api/docs/guides/function-calling
- OpenAI Responses API: https://developers.openai.com/api/reference/cli/resources/responses/methods/create